## ML Pipeline

**【Data Science Project】 Building a Machine Learning Pipeline for a Predictive Car Price Model with PySpark**

In [1]:
import os

# point java home to actual conda package reference
os.environ["JAVA_HOME"] = "/Users/andreasliistro/mambaforge/pkgs/openjdk-22.0.1-hbeb2e11_0/lib/jvm"

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import isnan, when, count, col, lit
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder

# init spark session
spark = SparkSession.builder.master("local[*]").config("spark.driver.memory", "4g").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/12/29 17:13:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/12/29 17:13:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# read previous written csv data
data = spark.read.csv("ML-pipeLine-car-prediction/data/vehicles_cleaned.csv", header=True, inferSchema=True)
# data = spark.read.csv("ML-pipeLine-car-prediction/data/vehicles.csv", header=True, inferSchema=True)

In [3]:
# show schema as reference
data.printSchema()

root
 |-- model: string (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- year: double (nullable = true)
 |-- condition: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- odometer: double (nullable = true)
 |-- title_status: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- drive: string (nullable = true)
 |-- size: string (nullable = true)
 |-- type: string (nullable = true)
 |-- paint_color: string (nullable = true)
 |-- description: string (nullable = true)
 |-- state: string (nullable = true)
 |-- posting_date: string (nullable = true)



In [13]:
# # convert types and clean data
# data = data.withColumn("price", data["price"].cast("double"))
# data = data.na.drop(subset=["price"])

# data = data.withColumn("year", data["year"].cast("double"))
# # data = data.withColumn("fuel", data["fuel"].cast("double"))
# data = data.withColumn("odometer", data["odometer"].cast("double"))
# data.printSchema()

root
 |-- id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- year: double (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- odometer: double (nullable = true)
 |-- title_status: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- type: string (nullable = true)
 |-- paint_color: string (nullable = true)
 |-- description: string (nullable = true)
 |-- state: string (nullable = true)
 |-- posting_date: string (nullable = true)



In [4]:
# show statisctics
data.describe().toPandas().transpose()

24/12/29 17:13:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
24/12/29 17:13:52 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


,0,1,2,3,4
summary,count,mean,stddev,min,max
model,418875,1787.3356730426565,3000.226205206505,"""1500 4wd crew cab 140.5"""" rebel""",♿ vmi
manufacturer,400570,2013.0827067669172,2.4357250295035895,2010,volvo
id,418875,7.311451577137132E9,4476911.55792221,* Images,✅ Mal crédito
price,416601,75255.42155229674,1.2325434572704755E7,-158.030906,3.736928711E9
year,410678,2011.4946600499661,8.966693643802264,1900.0,2022.0
condition,418875,None,None,excellent,unknown
fuel,418875,None,None,diesel,unknown
odometer,407358,96725.64597811851,197202.2130906528,0.0,1.0E7
title_status,405385,324.46937506426735,213.5919738551412,-116.56152,salvage


### Build the ML pipeline

Combining all numeric columns into feature vector. The column `price` is used as target column

In [5]:
#assemble all numeric columns into one vector of features
assembler = VectorAssembler(inputCols=[
  'year',
  # 'fuel',
  'odometer',
  # 'posting_date',
], outputCol='Attributes', handleInvalid="skip")

#create a regressor to predict car price
regressor = RandomForestRegressor(featuresCol='Attributes', labelCol='price')

#create pipeline
pipeline = Pipeline(stages=[assembler, regressor])

#save pipeline
pipeline.write().overwrite().save('pipeline')

### Perform cross-validation

In [6]:
#load pipeline
pipelineModel = Pipeline.load('pipeline')

#build paramgrid
paramGrid = ParamGridBuilder().addGrid(regressor.numTrees, [100, 500]).build()

#build crossvalidator
crossval = CrossValidator(estimator=pipelineModel,
                          estimatorParamMaps=paramGrid,
                          evaluator=RegressionEvaluator(labelCol='price'), #price is the column we want to predict
                          numFolds=10)

### Split data set into train / test

In [7]:
#train test split
train_data, test_data = data.randomSplit([0.8, 0.2], seed=123)

#fit
cvModel = crossval.fit(train_data)

#extract best model and view all the stages of the pipeline that our data went through
bestModel = cvModel.bestModel
for x in range(len(bestModel.stages)):
  print(bestModel.stages[x])

24/12/29 17:14:32 WARN DAGScheduler: Broadcasting large task binary with size 1022.3 KiB
24/12/29 17:14:35 WARN DAGScheduler: Broadcasting large task binary with size 1700.7 KiB
24/12/29 17:14:51 WARN DAGScheduler: Broadcasting large task binary with size 1008.7 KiB
24/12/29 17:14:54 WARN DAGScheduler: Broadcasting large task binary with size 1675.3 KiB
24/12/29 17:15:09 WARN DAGScheduler: Broadcasting large task binary with size 1022.3 KiB
24/12/29 17:15:12 WARN DAGScheduler: Broadcasting large task binary with size 1694.0 KiB
24/12/29 17:15:27 WARN DAGScheduler: Broadcasting large task binary with size 1019.0 KiB
24/12/29 17:15:29 WARN DAGScheduler: Broadcasting large task binary with size 1675.6 KiB
24/12/29 17:15:44 WARN DAGScheduler: Broadcasting large task binary with size 1022.3 KiB
24/12/29 17:15:47 WARN DAGScheduler: Broadcasting large task binary with size 1704.2 KiB
24/12/29 17:16:03 WARN DAGScheduler: Broadcasting large task binary with size 1045.9 KiB
24/12/29 17:16:06 WAR

VectorAssembler_57ed6293d220
RandomForestRegressionModel: uid=RandomForestRegressor_49e1f6b0680a, numTrees=500, numFeatures=2


In [8]:
#transform the test set (use cvModel as it knows to pick the best model to use)
pred = cvModel.transform(test_data)
pred.select('price', 'prediction').show()

+-------+------------------+
|  price|        prediction|
+-------+------------------+
|72995.0| 36062.80714451132|
|72995.0| 36062.80714451132|
|38997.0| 20507.41590029137|
|38997.0| 20507.41590029137|
|21997.0|  223284.377848976|
|19997.0| 52860.02440186822|
|    0.0| 27083.60501312537|
|    0.0| 27083.60501312537|
|74997.0|25431.131839174453|
|74997.0|25431.131839174453|
|74997.0|25431.131839174453|
|44997.0| 25593.77187242266|
|44997.0| 25593.77187242266|
|25997.0|  72109.9476114513|
|45900.0|42377.976069402524|
|25997.0|41729.364050937074|
|24997.0| 15876.21300165887|
|25997.0|26136.198207608857|
|53997.0| 25814.83342418884|
|    0.0|18293.870586136138|
+-------+------------------+
only showing top 20 rows



In [9]:
#evaluate
eval = RegressionEvaluator(labelCol='price')

#get rmse
rmse = eval.evaluate(pred)

#get mse
mse = eval.evaluate(pred, {eval.metricName:'mse'})

#get mae
mae = eval.evaluate(pred, {eval.metricName:'mae'})

#get r2
r2 = eval.evaluate(pred, {eval.metricName:'r2'})

#print
print('RMSE: %3f' %rmse)
print('MSE: %3f' %mse)
print('MAE: %3f' %mae)
print('R2: %3f' %r2)

RMSE: 3508120.557223
MSE: 12306909844007.949219
MAE: 90130.187695
R2: -0.007587
